In [ ]:
import visualizer

In [ ]:
import solvers.DWave_solver as DWave_solver
import map
import pathFormulation
import config.parser as config_parser
import QUBOBuilder
import config.hdf5parser as h5parser

In [ ]:
conf = config_parser.load_config("config/config.yaml", sections=["penalty_sets"])

solver_conf = config_parser.load_config("config/solvers.yaml")

map_conf = h5parser.load_map_from_hdf5("maps/synthetic/3x3/no_obs3x3_ter.h5")
materials_data = config_parser.load_config("config/materials.yaml")["materials"]
problem_conf = config_parser.load_config("maps/synthetic/3x3/no_obs3x3_ter.yaml", sections=["problems"])

unique_problem = problem_conf["problems"]["baseline"]

grid = map.Grid.from_hdf5_data(map_conf, materials_data)

problem = pathFormulation.PathfindingProblem.from_grid_dict(grid, unique_problem)

In [ ]:
penalties_conf = conf["penalty_sets"]["ter"]
builder = QUBOBuilder.QUBOBuilder(problem, penalties=penalties_conf, name="standard")

In [ ]:
solver = DWave_solver.QUBOSolver(normalize_scale=2.0, num_reads=15)
builder.reset_problem()
builder.build()
solution = solver.solve_qubo(builder)
path = solver.decode_path(solution["solution"], problem)
print("Path:", path)
print(f"Energy: {solver.total_energy(solution):.4f}")
print(builder.get_num_wires())

### Visualization

In [ ]:
visualizer = visualizer.QuantumRoboticsVisualizer(
        grid_size=(grid.M, grid.N), 
        title="Pathfinding Visualization",
        start_image_path="images/scooby.svg",
        goal_image_path="images/scoobysnack.svg",
        obstacle_image_path="images/ghost.png",
    )

step_fig = visualizer.create_step_by_step_plot(obstacles=grid.obstacles, path=path, start=problem.start, goal=problem.end, problem=problem)
visualizer.show(step_fig)